<a href="https://colab.research.google.com/github/David94240/project_1_billing_pipeline/blob/master/Collab/Collab_billing_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
import pandas as pd
from datetime import datetime
import logging
import os # Import os module to get absolute path

# Configuration des logs (une seule fois) using an absolute path
logging.basicConfig(
    filename=os.path.join(os.getcwd(), 'pipeline.log'),
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True # Force the configuration to apply even if logging is already configured
)

def load_data(filepath):
    """Charge le CSV en DataFrame."""
    return pd.read_csv(filepath, parse_dates=['date_facture'])

def clean_data(df):
    """
    Nettoie les données de facturation.
    """
    # Log initial count of duplicates in the incoming DataFrame
    initial_duplicates = df.duplicated(subset=['id_facture'], keep=False).sum()
    logging.info(f"Début du nettoyage: {initial_duplicates} doublons initiaux détectés sur id_facture avant tout traitement.")

    # 1. Compter les montants négatifs/manquants AVANT suppression
    montants_invalides = df[df['montant_HT'].isna() | (df['montant_HT'] < 0)]
    logging.info(f"Suppression de {len(montants_invalides)} lignes avec montants invalides.")

    # 2. Dates manquantes
    # Convertir explicitement les chaînes vides ou avec seulement des espaces en pd.NaT
    df['date_facture'] = df['date_facture'].replace(r'^\s*$', pd.NaT, regex=True)
    date_facturation_empty = df[df['date_facture'].isna()]
    logging.info(f"Nombre de lignes avec date_facture vide : {len(date_facturation_empty)}")
    df['date_facture'] = df['date_facture'].fillna(pd.Timestamp(datetime.now().date()))
    #logging.info(f"Remplacement de {df['date_facture'].isna().sum()} dates manquantes par la date du jour.")

    # 3. Suppression des montants HT invalides
    montant_HT_invalid = df[df['montant_HT'].isna() | (df['montant_HT'] < 0)]
    df = df[(df['montant_HT'].notna()) & (df['montant_HT'] >= 0)]
    logging.info(f"Suppression de {len(montant_HT_invalid)} lignes avec montants HT invalides.")

    # Suppression des taux_TVA non renseigner et mis à jour des logs
    taux_tva_missing_count = df['taux_TVA'].isna().sum() # Compter avant la suppression
    df = df[df['taux_TVA'].notna()]
    logging.info(f"Suppression de {taux_tva_missing_count} lignes avec taux_TVA non renseigner.")

    # 4. Normalisation des codes d'actes
    df['code_acte'] = df['code_acte'].str.replace('-', '').str.upper()
    logging.info(f"Normalisation des codes d'actes en majuscules et sans tirets. Nombre total d'actes uniques après normalisation : {df['code_acte'].nunique()}. ")

    # 5. Suppression des doublons
    doublons = df.duplicated(subset=['id_facture'], keep='first').sum() # Changed keep=False to keep='first'
    df = df.drop_duplicates(subset=['id_facture'], keep='first')
    logging.info(f"Suppression de {doublons} doublons sur id_facture.")

    # 6. Ajout d'une colonne calcul du montant_TTC
    df['montant_TTC'] = df['montant_HT'] * (1 + df['taux_TVA']).round(2)

    # 7. Ajout d'une colonne calcul du montant_TTC avec 2 chiffres après la virgule + suppression de la colonne montant_TTC
    df['montant_TTC_2C'] = (df['montant_TTC']).round(2)
    df = df.drop('montant_TTC', axis=1) # Supprime la colonne 'montant_TTC' du DataFrame

    # 8. Verifie si le montant TTC > HT avec un boolean
    df['montant_HT_sup_TTC'] = df['montant_HT'] > df['montant_TTC_2C']

    # Comptage du nombre de fasle qui sont à corriger :
    nombre_true = df['montant_HT_sup_TTC'].value_counts().get(True, 0)
    logging.info(f"Nombre de 'True' dans la colonne montant_HT_sup_TTC : {nombre_true}")

    return df

if __name__ == "__main__":
    input_file = "factures_brutes.csv"
    output_file = "factures_propres.csv"

    print(f"Current Working Directory: {os.getcwd()}") # Debugging line
    logging.info("Pipeline script started.")

    # Chargement et nettoyage
    df_brut = load_data(input_file)
    logging.info(f"Chargement de {len(df_brut)} lignes depuis {input_file}.")

    df_propre = clean_data(df_brut)
    df_propre.to_csv(output_file, index=False)
    logging.info(f"Pipeline terminé. Résultat sauvegardé dans {output_file}.")

    logging.shutdown()

Current Working Directory: /content


In [32]:
import pandas as pd
import numpy as np

# Création d'un DataFrame d'exemple avec des doublons et des valeurs manquantes
data = {
    'id_facture': ['FACT-001', 'FACT-002', 'FACT-001', 'FACT-003', 'FACT-002'],
    'taux_TVA': [0.2, 0.1, np.nan, 0.2, 0.2],
    'montant_HT': [100, 200, 110, 300, 210]
}
df_example = pd.DataFrame(data)

print("DataFrame original:")
display(df_example)

# Application de drop_duplicates avec keep='first'
df_cleaned = df_example.drop_duplicates(subset=['id_facture'], keep='first')

print("\nDataFrame après suppression des doublons (keep='first'):")
display(df_cleaned)

DataFrame original:


,id_facture,taux_TVA,montant_HT
0,FACT-001,0.2,100
1,FACT-002,0.1,200
2,FACT-001,NaN,110
3,FACT-003,0.2,300
4,FACT-002,0.2,210



DataFrame après suppression des doublons (keep='first'):


,id_facture,taux_TVA,montant_HT
0,FACT-001,0.2,100
1,FACT-002,0.1,200
3,FACT-003,0.2,300


In [28]:
import os
print(os.listdir('.'))

['.config', '.ipynb_checkpoints', 'factures_propres.csv', 'factures_brutes.csv', 'pipeline.log', 'sample_data']
